# 🌾 NeuralHive25 — The Kaggle Competition  
### Dataset Creation & Context

---

## 🧩 Origin of the Dataset

The foundation for this competition’s dataset comes from the publicly available **[Crop Production in India Dataset](https://www.kaggle.com/datasets/abhinand05/crop-production-in-india)** on Kaggle.

This dataset contains extensive agricultural information from multiple Indian states and districts across several crop years.  
It includes attributes such as:

- **State_Name**
- **District_Name**
- **Crop_Year**
- **Season**
- **Crop**
- **Area (in hectares)**
- **Production (in tonnes)**

---

## 🌱 Context

Agriculture remains the backbone of the Indian economy, contributing significantly to GDP and livelihoods.  
The dataset captures patterns and trends in crop yields across **different regions, seasons, and crop types**.

The **goal** of this dataset is to predict **crop production** given contextual and environmental variables.  
In this competition, participants are expected to leverage **machine learning and data preprocessing techniques** to accurately model production levels.

---

## 🎯 Objective

> Predict the `Production` of a given crop for a particular state, district, year, and season, based on various input features.

However, to make the challenge more realistic, robust, and competitive for the **Neural Hive AI/ML Club recruitment**,  
we’ve significantly **transformed and enhanced** the raw dataset — adding noise, synthetic features, and outliers to simulate real-world data imperfections.

---

Next, we’ll go step-by-step through **how we cleaned, engineered, and modified** the original dataset to create the final competition dataset:  
`TheKaggleCompetition.csv` 🌾


## ⚙️ Step 1 — Importing Required Libraries

To begin, we import the essential Python libraries used for **data processing, numerical operations, dataset splitting, and file management**.

- **pandas** → For data manipulation and creating structured DataFrames.  
- **numpy** → For performing numerical operations, adding noise, and generating random values.  
- **sklearn.model_selection.train_test_split** → To split our dataset into training and testing subsets efficiently.  
- **pathlib.Path** → For managing file paths and directories in a clean, OS-independent way.

These libraries together form the foundation for reading the raw dataset, transforming it, and saving all processed outputs into organized folders for the competition.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path

## 🧱 Step 2 — Setting Up Configuration and Directory Structure

Before processing the dataset, we define a few key configuration variables and create the necessary folder structure.

- **RANDOM_SEED = 42**  
  Ensures that all random operations (like shuffling, splitting, and noise addition) are reproducible.  
  Setting a fixed seed guarantees consistent results each time the script runs.

- **BASE_DIR**  
  The base working directory for all input and output files.

- **RAW_FILE**  
  Points to the original raw dataset — in this case, located at `raw/raw.csv`.

- **CLEAN_FILE, KAGGLE_DIR, LEADERBOARD_DIR**  
  Define the output file and folder paths:
  - `TheKaggleCompetition.csv` → final processed master dataset  
  - `kaggle/` → contains `train.csv`, `test.csv`, and `sample_submission.csv`  
  - `leaderboard/` → contains `public_leaderboard.csv` and `private_leaderboard.csv`

Finally, we create these directories using `mkdir(parents=True, exist_ok=True)` to ensure the structure exists before saving any files.

In [2]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

BASE_DIR = Path("./")
RAW_FILE = BASE_DIR / "raw/raw.csv"

# Output directories
CLEAN_FILE = BASE_DIR / "TheKaggleCompetition.csv"
KAGGLE_DIR = BASE_DIR / "kaggle"
LEADERBOARD_DIR = BASE_DIR / "leaderboard"

KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
LEADERBOARD_DIR.mkdir(parents=True, exist_ok=True)

## 📥 Step 3 — Loading and Cleaning the Raw Dataset

We begin by loading the original dataset from the path defined earlier (`raw/raw.csv`) using **pandas**.

To ensure data quality and consistency before further processing, a few basic cleaning steps are applied:

1. **Remove Missing Values:**  
   Any rows with missing entries in critical columns — `Area` or `Production` — are dropped, as they directly affect model training and target prediction.

2. **Filter Invalid Records:**  
   Rows where `Area` is zero or negative are removed since such entries don’t represent meaningful agricultural data.

3. **Reset Index:**  
   The DataFrame index is reset after cleaning to maintain a continuous sequence.

Finally, the script prints the total number of valid records remaining after preprocessing.  
In our case, the cleaned dataset contains **242,361 data points**, forming the foundation for all subsequent transformations.

In [3]:
print("📥 Loading raw dataset...")
df = pd.read_csv(RAW_FILE)

# Basic cleanup
df = df.dropna(subset=["Area", "Production"])
df = df[df["Area"] > 0]
df = df.reset_index(drop=True)

n = len(df)
print(f"🧮 Records loaded: {n}")

📥 Loading raw dataset...
🧮 Records loaded: 242361


## 🌾 Step 4 — Adding Synthetic and Misleading Features

To make the dataset richer, more realistic, and challenging, we generate a set of **synthetic features** that mimic real-world agricultural factors.  
These additional columns help simulate the complexity of actual data collected from diverse farming regions.

### 🔹 Semi-Meaningful Synthetic Features
- **Rainfall_mm:** Simulated rainfall values using a normal distribution (mean ≈ 950 mm) and scaled relative to the crop area — larger areas tend to receive or require more rainfall.  
- **Temperature_C:** Represents average growing-season temperature (mean ≈ 28°C) with slight random variation.  
- **Fertilizer_kg_per_ha:** Randomized fertilizer usage per hectare, between 100–350 kg.  
- **Soil_Quality:** A normalized score (0.2–0.95) indicating general soil fertility.

### 🔹 Misleading or Redundant Features
To make the competition more analytical and prevent overfitting, we add features that **seem relevant but don’t contribute much predictive value**:
- **Irrigation_Index:** A derived value loosely based on area, but with random scaling.  
- **Govt_Subsidy_Score:** Correlated with production, but with noise — easy to misuse for naive models.  
- **Market_Accessibility:** Random normally distributed values, completely uncorrelated.  
- **Random_Noise_Factor:** Pure noise, designed to test feature selection and regularization techniques.

These additions ensure participants must rely on solid **feature engineering and model validation** instead of blindly trusting all variables.

In [4]:
# Add semi-meaningful synthetic features
df["Rainfall_mm"] = np.random.normal(950, 200, n) * (1 + df["Area"] / df["Area"].max())
df["Temperature_C"] = np.random.normal(28, 3, n)
df["Fertilizer_kg_per_ha"] = np.random.uniform(100, 350, n)
df["Soil_Quality"] = np.random.uniform(0.2, 0.95, n)

# Misleading or redundant features
df["Irrigation_Index"] = df["Area"] * np.random.uniform(0.9, 1.1, n)
df["Govt_Subsidy_Score"] = df["Production"] * np.random.uniform(0.8, 1.2, n)
df["Market_Accessibility"] = np.random.normal(0, 1, n)
df["Random_Noise_Factor"] = np.random.uniform(0, 1, n)


## 🌫️ Step 5 — Introducing Controlled Gaussian Noise

To make the dataset more realistic and reflective of real-world inconsistencies, we introduce **Gaussian (normal) noise** into key numerical features.

For each selected column —  
`Area`, `Production`, `Rainfall_mm`, `Temperature_C`, and `Fertilizer_kg_per_ha` —  
we slightly perturb the values by multiplying them with random factors drawn from a normal distribution centered at 0, with a standard deviation of 0.05.

This results in approximately **±5% variation** around the original values, effectively simulating:
- Measurement errors in agricultural surveys  
- Natural variability in environmental and crop data  
- Slight randomness that challenges overly deterministic models

By doing this, we ensure the dataset behaves more like **real-world data**, forcing participants to build models that can generalize rather than memorize.

In [5]:
for col in ["Area", "Production", "Rainfall_mm", "Temperature_C", "Fertilizer_kg_per_ha"]:
    df[col] *= (1 + np.random.normal(0, 0.05, n))  # ±5% Gaussian noise

## 🚨 Step 6 — Injecting Outliers into the Dataset

To simulate real-world data irregularities and make the problem more challenging, we deliberately introduce **outliers** into the target variable — `Production`.

### 🔹 How It Works
- We randomly select about **2% of the dataset** (`0.02 * n`) as potential outlier rows.  
- For these selected records, the `Production` values are multiplied by a random factor between **1.5× and 3.0×**.

This creates instances where production values are **abnormally high**, mimicking real-world phenomena such as:
- Exceptional yield due to favorable weather  
- Data entry or reporting errors  
- Rare but impactful agricultural conditions

These outliers ensure participants must handle data cleaning, robust modeling, or outlier-resistant algorithms to achieve good leaderboard performance.

In [6]:
outlier_idx = np.random.choice(df.index, int(0.02 * n), replace=False)
df.loc[outlier_idx, "Production"] *= np.random.uniform(1.5, 3.0, len(outlier_idx))

## 🕳️ Step 7 — Introducing Missing Values

Real-world datasets are rarely perfect — sensor errors, manual data entry mistakes, and incomplete records are common.  
To replicate these conditions, we intentionally introduce **missing values** in selected numerical features.

### 🔹 Implementation Details
- For each of the following columns:  
  `Rainfall_mm`, `Temperature_C`, `Fertilizer_kg_per_ha`, and `Soil_Quality`,  
  around **3% of the entries** (`frac=0.03`) are randomly set to `NaN`.

- The `random_state` is fixed using our global `RANDOM_SEED` for reproducibility.

This step ensures participants must handle **data imputation and preprocessing** effectively — an essential skill in real-world machine learning pipelines.  
Ignoring these missing values would lead to biased or incomplete models, so proper handling becomes part of the challenge.

In [7]:
for col in ["Rainfall_mm", "Temperature_C", "Fertilizer_kg_per_ha", "Soil_Quality"]:
    missing_idx = df.sample(frac=0.03, random_state=RANDOM_SEED).index
    df.loc[missing_idx, col] = np.nan

## 🔢 Step 8 — Adding Serial Numbers

To maintain consistent tracking of each record throughout the dataset transformation pipeline,  
we add a new column called **`srno`** (serial number) at the beginning of the dataframe.

### 🔹 Purpose
- Ensures every row has a **unique identifier**.  
- Makes it easier to **map records** between the train, test, and leaderboard splits.  
- Simplifies validation, debugging, and leaderboard mapping later in the competition.

Each record receives a unique `srno` starting from **1** up to the total number of rows in the dataset.

In [8]:
df.insert(0, "srno", range(1, len(df) + 1))

## 🔀 Step 9 — Shuffling the Dataset

To eliminate any potential ordering bias (for example, rows grouped by state, crop, or year),  
we **randomly shuffle** the entire dataset.

### 🔹 Implementation
- We use `sample(frac=1)` to randomly reorder **all rows** in the dataset.  
- The `random_state` ensures **reproducibility**, so the shuffle can be recreated exactly.

Shuffling ensures the **train-test splits** are unbiased and representative of the overall data distribution,  
preventing models from learning unintended patterns based on dataset order.

In [9]:
df = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

## ✂️ Step 10 — Splitting into Train and Test Sets

Next, we divide the shuffled dataset into **training** and **testing** subsets.

### 🔹 Implementation Details
- **Training Set:** 75% of the data — used by participants to train and validate their models.  
- **Test Set:** 25% of the data — used for final leaderboard evaluation.  
- The split is controlled by the same `RANDOM_SEED` for full reproducibility.

### 🔹 Why It Matters
This separation ensures that:
- The model evaluation remains **fair and unbiased**.  
- Participants can’t overfit to known targets.  
- Only unseen data (the test set) determines leaderboard performance.

In [10]:
train_df, test_df = train_test_split(df, test_size=0.25, random_state=RANDOM_SEED)

## 🧩 Step 11 — Hiding the Target Variable for the Test Set

To ensure fair competition and prevent data leakage, we remove the target variable — **`Production`** —  
from the test dataset before sharing it with participants.

### 🔹 Implementation
- The **training dataset (`train_df`)** retains all columns, including the target variable `Production`.  
- The **test dataset (`test_nolabel`)** excludes `Production`, since participants will need to predict it.

### 🔹 Why This Step Is Crucial
- Prevents **data leakage**, ensuring participants can’t peek at the true target values.  
- Simulates a **real-world prediction scenario**, where future outcomes are unknown.  
- The hidden targets are later used internally to compute **leaderboard scores**.

In [11]:
# Remove target for test data
test_nolabel = test_df.drop(columns=["Production"]).reset_index(drop=True)
train_df = train_df.reset_index(drop=True)

## 🏁 Step 12 — Creating Public and Private Leaderboards

To evaluate participants fairly and discourage overfitting to the leaderboard,  
we split the **test set** into two distinct parts:

### 🔹 Implementation Details
- **Public Leaderboard:** 40% of the test set — used for live leaderboard updates during the competition.  
- **Private Leaderboard:** 60% of the test set — kept hidden until the competition ends, used for final ranking.  

Both splits are created using `train_test_split()` with a fixed random seed to ensure reproducibility.

### 🔹 Ground Truth Storage
For both leaderboard sets, we store only:
- `srno` — unique record identifier.  
- `Production` — true target value (for internal scoring only).

These files (`public_truth` and `private_truth`) will **never be shared** with participants.  
They are used by the organizers to compute the **public and private leaderboard scores** after submission

In [12]:
public_df, private_df = train_test_split(test_df, test_size=0.6, random_state=RANDOM_SEED)

# Keep srno + ground truth for scoring
public_truth = public_df[["srno", "Production"]].sort_values("srno").reset_index(drop=True)
private_truth = private_df[["srno", "Production"]].sort_values("srno").reset_index(drop=True)

## 📤 Step 13 — Generating the Sample Submission File

To help participants understand the expected submission format,  
we create a **sample submission file** that mimics the final prediction structure.

### 🔹 Implementation Details
- Each row corresponds to a unique test record identified by `srno`.  
- The column `predicted_value` represents the model’s prediction for the `Production` value.  
- Instead of zeros, we fill this column with **randomized values** within a realistic range:
  - Between **50% of the minimum** and **110% of the maximum** observed `Production` in the training set.
- Values are rounded to two decimal places for consistency.

### 🔹 Purpose
This file serves as a **template** for participants to structure their submissions correctly.  
They will replace these random predictions with their model outputs before uploading to Kaggle.

In [13]:
sample_submission = pd.DataFrame({
    "srno": test_nolabel["srno"],
    "predicted_value": np.random.uniform(
        low=train_df["Production"].min() * 0.5,
        high=train_df["Production"].max() * 1.1,
        size=len(test_nolabel)
    ).round(2)
})

### ✅ Completion
Once the files are saved:
- You can upload the `train.csv`, `test.csv`, and `sample_submission.csv` to Kaggle for participants.  
- Keep the leaderboard CSVs private for evaluation after submissions.

This marks the **end of dataset preparation** for  
🏆 **NeuralHive25 — The Kaggle Competition**.

Your dataset is now fully ready for launch! 🚀

In [14]:
print("💾 Saving outputs...")

df.to_csv(CLEAN_FILE, index=False)
train_df.to_csv(KAGGLE_DIR / "train.csv", index=False)
test_nolabel.to_csv(KAGGLE_DIR / "test.csv", index=False)
sample_submission.to_csv(KAGGLE_DIR / "sample_submission.csv", index=False)
public_truth.to_csv(LEADERBOARD_DIR / "public_leaderboard.csv", index=False)
private_truth.to_csv(LEADERBOARD_DIR / "private_leaderboard.csv", index=False)

print("\n✅ Dataset generation complete!")
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_nolabel.shape}")
print(f"Public LB: {public_truth.shape} | Private LB: {private_truth.shape}")
print(f"Output: {BASE_DIR.resolve()}")

💾 Saving outputs...

✅ Dataset generation complete!
Train shape: (181770, 16)
Test shape: (60591, 15)
Public LB: (24236, 2) | Private LB: (36355, 2)
Output: /home/sidd/Desktop/kaggle-comp/dataset
